
## ETL/Gold/01 - Dimensiones Type 0 (fecha) y SCD Type 1
##   dim_fecha (T0)  +  dim_organizacion, dim_task,
##   dim_libreria, dim_licencia, dim_tag (SCD1)
## Implementacion con spark.sql (canonica en src/DDL/gold/*)

In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = get_param("catalog", "pf")
S_MODELO = f"{CATALOG}.silver.modelos"
S_TAG    = f"{CATALOG}.silver.model_tag"

In [0]:
%py

spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_dates AS
    SELECT
        CAST(DATE_FORMAT(d, 'yyyyMMdd') AS BIGINT) AS fecha_id,
        d AS fecha,
        YEAR(d) AS anio,
        QUARTER(d) AS trimestre,
        MONTH(d) AS mes,
        INITCAP(DATE_FORMAT(d, 'MMMM')) AS nombre_mes,
        DAYOFMONTH(d) AS dia,
        DATE_FORMAT(d, 'EEEE') AS dia_semana,
        (DAYOFWEEK(d) IN (1, 7)) AS es_fin_de_semana
    FROM (SELECT EXPLODE(SEQUENCE(
        TO_DATE('2020-01-01'), TO_DATE('2032-12-31'), INTERVAL 1 DAY)) AS d)
""")

spark.sql("""
    MERGE INTO pf.gold.dim_fecha AS target
    USING v_dates AS src
    ON target.fecha_id = src.fecha_id
    WHEN NOT MATCHED THEN
      INSERT (fecha_id, fecha, anio, trimestre, mes, nombre_mes, dia, dia_semana,
              es_fin_de_semana, _createdAt)
      VALUES (src.fecha_id, src.fecha, src.anio, src.trimestre, src.mes, src.nombre_mes,
              src.dia, src.dia_semana, src.es_fin_de_semana, CURRENT_TIMESTAMP())
""")

In [0]:
%py

spark.sql(f"""
    MERGE INTO pf.gold.dim_organizacion AS t
    USING (
        SELECT org_id AS org_id,
               org_id AS org_nombre,
               COUNT(DISTINCT model_id) AS num_modelos_ref
        FROM {S_MODELO}
        GROUP BY org_id
    ) AS s
    ON t.org_id = s.org_id
    WHEN MATCHED THEN
        UPDATE SET t.org_nombre = s.org_nombre,
                   t.num_modelos_ref = s.num_modelos_ref
    WHEN NOT MATCHED THEN
        INSERT (org_id, org_nombre, num_modelos_ref, _createdAt)
        VALUES (s.org_id, s.org_nombre, s.num_modelos_ref, CURRENT_TIMESTAMP())
""")

In [0]:
%py

spark.sql(f"""
    MERGE INTO pf.gold.dim_task AS t
    USING (
        SELECT pipeline_tag AS pipeline_tag,
               CASE pipeline_tag
                   WHEN 'text-generation' THEN 'Generacion de texto (LLM)'
                   WHEN 'text-classification' THEN 'Clasificacion de texto'
                   WHEN 'sentence-similarity' THEN 'Similitud de frases / embeddings'
                   WHEN 'text-embedding' THEN 'Embeddings de texto'
                   WHEN 'fill-mask' THEN 'Enmascarado de texto'
                   WHEN 'token-classification' THEN 'NER / etiquetado de tokens'
                   WHEN 'translation' THEN 'Traduccion'
                   WHEN 'summarization' THEN 'Resumen'
                   WHEN 'question-answering' THEN 'Pregunta-respuesta'
                   WHEN 'automatic-speech-recognition' THEN 'Reconocimiento de voz'
                   WHEN 'image-classification' THEN 'Clasificacion de imagenes'
                   ELSE 'Otra'
               END AS descripcion
        FROM {S_MODELO}
        WHERE pipeline_tag IS NOT NULL
        GROUP BY pipeline_tag
    ) AS s
    ON t.pipeline_tag = s.pipeline_tag
    WHEN MATCHED THEN
        UPDATE SET t.descripcion = s.descripcion
    WHEN NOT MATCHED THEN
        INSERT (pipeline_tag, descripcion, _createdAt)
        VALUES (s.pipeline_tag, s.descripcion, CURRENT_TIMESTAMP())
""")

In [0]:
%py

# spark.sql(f"""
#     MERGE INTO pf.gold.dim_libreria AS t
#     USING (
#         SELECT 
#                UPPER(library_name) AS library_name,
#                library_name AS descripcion
#         FROM {S_MODELO}
#         WHERE library_name IS NOT NULL
#         GROUP BY UPPER(library_name)
#     ) AS s
#     ON t.library_name = s.library_name
#     WHEN MATCHED THEN
#         UPDATE SET t.descripcion = s.descripcion
#     WHEN NOT MATCHED THEN
#         INSERT (library_name, descripcion, _createdAt)
#         VALUES (s.library_name, s.descripcion, CURRENT_TIMESTAMP())
# """)
spark.sql(f"""

    MERGE INTO pf.gold.dim_libreria AS t

    USING (

        SELECT
            UPPER(library_name) AS library_name,
            any_value(library_name) AS descripcion

        FROM {S_MODELO}

        WHERE library_name IS NOT NULL

        GROUP BY UPPER(library_name)

    ) AS s

    ON t.library_name = s.library_name

    WHEN MATCHED THEN

        UPDATE SET
            t.descripcion = s.descripcion

    WHEN NOT MATCHED THEN

        INSERT (
            library_name,
            descripcion,
            _createdAt
        )

        VALUES (
            s.library_name,
            s.descripcion,
            CURRENT_TIMESTAMP()
        )

""")

In [0]:
%py

spark.sql(f"""
    MERGE INTO pf.gold.dim_licencia AS t
    USING (
        SELECT COALESCE(license_tag, 'sin_licencia') AS license_tag
        FROM {S_MODELO}
        GROUP BY COALESCE(license_tag, 'sin_licencia')
    ) AS s
    ON t.license_tag = s.license_tag
    WHEN MATCHED THEN
        UPDATE SET t.descripcion = CASE
                        WHEN s.license_tag IN ('license:apache-2.0','license:mit') THEN 'Open-weight permisiva'
                        WHEN s.license_tag = 'sin_licencia' THEN 'Sin licencia declarada'
                        ELSE 'Licencia de uso restringido'
                    END,
                   t.es_permissiva = (s.license_tag IN ('license:apache-2.0','license:mit'))
    WHEN NOT MATCHED THEN
        INSERT (license_tag, descripcion, es_permissiva, _createdAt)
        VALUES (s.license_tag,
                CASE WHEN s.license_tag IN ('license:apache-2.0','license:mit') THEN 'Open-weight permisiva'
                     WHEN s.license_tag = 'sin_licencia' THEN 'Sin licencia declarada'
                     ELSE 'Licencia de uso restringido' END,
                (s.license_tag IN ('license:apache-2.0','license:mit')),
                CURRENT_TIMESTAMP())
""")

In [0]:
%py

spark.sql(f"""
    MERGE INTO pf.gold.dim_tag AS t
    USING (
        SELECT DISTINCT
            tag,
            CASE WHEN es_license THEN 'license'
                 WHEN es_dataset THEN 'dataset'
                 WHEN es_arxiv   THEN 'arxiv'
                 ELSE 'generico' END AS tipo_tag
        FROM {S_TAG}
    ) AS s
    ON t.tag = s.tag
    WHEN MATCHED THEN
        UPDATE SET t.tipo_tag = s.tipo_tag
    WHEN NOT MATCHED THEN
        INSERT (tag, tipo_tag, _createdAt)
        VALUES (s.tag, s.tipo_tag, CURRENT_TIMESTAMP())
""")

In [0]:
%py

for dim in ["dim_organizacion", "dim_task", "dim_libreria", "dim_licencia", "dim_tag"]:
    spark.sql(f"SELECT COUNT(*) AS n, '{dim}' AS dim FROM pf.gold.{dim}").show()